In [ ]:
pip install findspark

In [1]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1521

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


Successfully connected to HYPE on port 1521.


In [2]:
import os
import findspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") # Added confirmation message

Findspark initialized successfully.


In [3]:
import os
import findspark
from pyspark.sql import SparkSession
import pyspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") 

# Define the paths to your local JAR files, now with the file:/// scheme
# Use forward slashes exclusively in the final path string
mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-13.2.1.jre11.jar"
osql_jar_path = r"file:///C:/spark_drivers/ojdbc17.jar"

spark = SparkSession.builder \
    .appName("SQLServerToosqlTransfer") \
    .config("spark.jars", f"{mssql_jar_path},{osql_jar_path}") \
    .getOrCreate()

print("Spark session created.")

# You can now proceed with your data transfer logic here
# For example:
# df = spark.read.format("jdbc").option(...)

Findspark initialized successfully.
Spark session created.


In [9]:
# SQL Server connection properties
# cannot make partition with char column
import time
sql_server_url = r"jdbc:sqlserver://HYPE\SQL_2022:1433;databaseName=sttwk_2025a_par;encrypt=true;trustServerCertificate=true"
sqlserver_properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "fetchSize": "1000"
}
# Start the timer
start_time = time.time()

# Parameters for parallel read (adjust as needed for your table and data)
# The partitionColumn must be a numeric, date, or timestamp column.
#partition_column = "week" 
#lower_bound = 1
#upper_bound = 13 # Max ID in your table, e.g.
#num_read_partitions = 13 # Read in 10 partitions
    #.option("url", sqlserver_jdbc_url) \
    #.option("partitionColumn", partition_column) \
    #.option("lowerBound", lower_bound) \
    #.option("upperBound", upper_bound) \
    #.option("numPartitions", num_read_partitions) \
sql_server_df = spark.read.format("jdbc") \
    .option("url", sql_server_url) \
    .option("dbtable", "v_sttwk_2025") \
    .option("user", sqlserver_properties["user"]) \
    .option("password", sqlserver_properties["password"]) \
    .option("driver", sqlserver_properties["driver"]) \
    .load()
# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")

# Optional: Check the number of partitions created
print(f"DataFrame has {sql_server_df.rdd.getNumPartitions()} partitions after read.")


Total time taken for transfer: 0.05 seconds
DataFrame has 1 partitions after read.


In [10]:
# 10 seconds
from pyspark.sql import functions as sf
# Start the timer
start_time = time.time()

# Option A: Repartition by the character column (hash partitioning)
# The data associated with each unique value in 'char_column' will go to a single partition.
df_to_write = sql_server_df.repartition(25, "wk")

# Option B: Repartition generally (even hash distribution)
# Use this if you don't care about co-locating data by the char column, 
# or if the data in 'your_char_column_name' is very skewed.
# df_to_write = sql_server_df.repartition(10) 

print(f"DataFrame has {df_to_write.rdd.getNumPartitions()} partitions before write.")

# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")

DataFrame has 25 partitions before write.
Total time taken for transfer: 28.85 seconds


In [ ]:
'''
df_to_write_renamed = df_to_write.withColumnRenamed("YEAR_COL", "year") \
                                         .withColumnRenamed("MONTH_COL", "month") \
                                         .withColumnRenamed("WEEK_COL", "week") \
                                         .withColumnRenamed("BRANCH_ID", "branch_id") \
                                         .withColumnRenamed("SP_ID", "sp_id") \
                                         .withColumnRenamed("CUSTD_ID", "custd_id") \
                                         .withColumnRenamed("PROD_ID", "prod_id") \
                                         .withColumnRenamed("SLS_ID", "sls_id") \
                                         .withColumnRenamed("NET_QTY", "net_qty") \
                                         .withColumnRenamed("NET_AMT", "net_amt") \
                                         .withColumnRenamed("GRS_QTY", "grs_qty") \
                                         .withColumnRenamed("GRS_AMT", "grs_amt") \
                                         .withColumnRenamed("RTN_QTY", "rtn_qty") \
                                         .withColumnRenamed("RTN_AMT", "rtn_amt")

print("successfully renamed.")
'''

In [11]:
# osql connection properties
# Start the timer
start_time = time.time()
# Connection details for osql (Destination)
oracle_host = "localhost"
oracle_port = "1521"
oracle_service_name = "XE" # Standard service name for XE
oracle_user = "system"        
oracle_password = "D@1ry$2014" 
oracle_table = "STTWK"        
oracle_driver_class = "oracle.jdbc.driver.OracleDriver" 

# This is the crucial line: Make sure the URL is formatted exactly like this:
oracle_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service_name}"


oracle_jdbc_url =  f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service_name}"
 
 
# Write the repartitioned DataFrame to the osql table
df_to_write.write.format("jdbc") \
    .option("url", oracle_url) \
    .option("dbtable", oracle_table) \
    .option("user", oracle_user) \
    .option("password", oracle_password) \
    .option("driver", oracle_driver_class) \
    .option("batchsize", "10000") \
    .mode("append") \
    .save()


print("Data successfully inserted into osql table using 52 parallel processes.")
# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")    

Data successfully inserted into osql table using 52 parallel processes.
Total time taken for transfer: 61.50 seconds


In [18]:
# Assuming 'spark' session is still active from Cell 1
# Start the timer
start_time = time.time()

# Connection details
osql_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service_name}"
osql_user = "system"
osql_password = "D@1ry$2014" 
oracle_driver_class = "oracle.jdbc.driver.OracleDriver"

# Define the full, raw SQL query.
# IMPORTANT: Remove the AS data_count alias, as Spark handles that internally now.
full_sql_query = "select yr,wk,sum(net_qty) as net_qty from sttwk group by yr,wk"

# Read the data using the 'query' option instead of 'dbtable'
df_sample = spark.read.format("jdbc").options(
    url=osql_url,
    query=full_sql_query,  # Use 'query' instead of 'dbtable'
    user=osql_user,
    password=osql_password,
    driver=oracle_driver_class
).load()

print("Displaying aggregated records from Oracle table 'sttwk':")

# Use .show() to display the records (ordering them in Spark)
df_sample.orderBy("yr", "wk").show(n=1000, truncate=False)

# To get the count of the sample:
print(f"Number of sample rows read: {df_sample.count()}")

# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")


Displaying aggregated records from Oracle table 'sttwk':
+----+---+------------------+
|YR  |WK |NET_QTY           |
+----+---+------------------+
|2022|01 |1176184.4970000000|
|2022|02 |1450239.3370000000|
|2022|03 |1337833.0700000000|
|2022|04 |1625286.4900000000|
|2022|05 |878449.0460000000 |
|2022|06 |1382949.0510000000|
|2022|07 |1459358.2600000000|
|2022|08 |1767098.0740000000|
|2022|09 |840785.0730000000 |
|2022|10 |1462618.2410000000|
|2022|11 |1452046.7130000000|
|2022|12 |1754371.8310000000|
|2022|13 |2204147.0730000000|
|2022|14 |978069.7020000000 |
|2022|15 |1163259.9000000000|
|2022|16 |1272122.0550000000|
|2022|17 |1565855.1780000000|
|2022|18 |125007.7630000000 |
|2022|19 |1255309.4240000000|
|2022|20 |1341211.9380000000|
|2022|21 |1611366.0010000000|
|2022|22 |838319.5850000000 |
|2022|23 |1436985.5590000000|
|2022|24 |1263733.2030000000|
|2022|25 |1288682.5930000000|
|2022|26 |1185235.8260000000|
|2022|27 |1111092.9400000000|
|2022|28 |939130.3910000000 |
|2022|29 |112